# processing zecmip data

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
# import matplotlib as mpl
# import matplotlib.ticker as mticker
import netCDF4
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.util import add_cyclic_point
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [ ]:
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# client

In [ ]:
from scipy import stats

In [ ]:
# mpl.rcParams['font.family'] = 'Droid Sans'
# mpl.rcParams['font.size'] = 12
# # Edit axes parameters
# mpl.rcParams['axes.linewidth'] = 1.5
# # Tick properties
# mpl.rcParams['xtick.major.size'] = 5
# mpl.rcParams['xtick.minor.size'] = 3
# mpl.rcParams['xtick.major.width'] = 1
# mpl.rcParams['xtick.direction'] = 'out'
# mpl.rcParams['ytick.major.size'] = 5
# mpl.rcParams['ytick.minor.size'] = 3
# mpl.rcParams['ytick.major.width'] = 1
# mpl.rcParams['ytick.direction'] = 'out'

### Functions needed for the analysis

In [ ]:
from functions import preproc_funcs as funcs

In [ ]:
from functions import xr_lowess

In [ ]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [ ]:
import glob
import multiprocessing as mp

In [ ]:
models_cdr = [
    'ACCESS-ESM1-5',
    # 'CAS-ESM2-0', # tas not available
    'CanESM5',
    'CESM2',
    'CNRM-ESM2-1',
    'MIROC-ES2L',
    'GFDL-ESM4',
    'NorESM2-LM',
    'UKESM1-0-LL',
]

In [ ]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        # model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [183]:
model = models_cdr[7]
model

'UKESM1-0-LL'

In [184]:
tas_pattern_cdr = f'/g/data/*/*/CMIP6/CDRMIP/*/{model}/1pctCO2-cdr/*/Amon/tas/*/*/*.nc'

In [185]:
co2_pattern_cdr = f'/g/data/*/*/CMIP6/CDRMIP/*/{model}/1pctCO2-cdr/*/Amon/co2/*/*/*.nc'

In [186]:
# '/g/data/oi10/replicas/CMIP6/C4MIP/CCCma/CanESM5/esm-1pct-brch-1000PgC/r1i1p2f1/Amon/'

In [187]:
pr_pattern_cdr = f'/g/data/*/*/CMIP6/CDRMIP/*/{model}/1pctpr-cdr/*/Amon/pr/*/*/*.nc'

In [188]:
tas_files_cdr = find_all_files(tas_pattern_cdr)

In [189]:
co2_files_cdr = find_all_files(co2_pattern_cdr)

In [190]:
pr_files_cdr = find_all_files(pr_pattern_cdr)

In [191]:
tas_files_cdr

{'UKESM1-0-LL_r1i1p1f': '/g/data/oi10/replicas/CMIP6/CDRMIP/MOHC/UKESM1-0-LL/1pctCO2-cdr/r1i1p1f2/Amon/tas/gn/v20200425/*.nc'}

In [192]:
import xesmf as xe

In [193]:
# temp_hist = xr.open_dataset(ts_files_hist['ACCESS-CM2'])
# temp_hist

In [194]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [195]:
from dask.diagnostics import ProgressBar

In [196]:
# global_regridder = xe.Regridder(xc.open_mfdataset(uas_files_hist['ACCESS-ESM1-5_r10i1p1f']).load(), ds_out, 'bilinear', periodic=True, ignore_degenerate=True)
# global_regridder

In [197]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_zecmip(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasetas
        var_file = tas_files_cdr[model_identifier]
        ds_var = xc.open_mfdataset(var_file, use_cftime=True, coords='minimal', compat='override')#.sel(time = slice('1850-01-01', '2015-01-01'))
        # ds_var = xc.open_mfdataset(var_file, use_cftime=True)#.sel(time = slice('1850-01-01', '2015-01-01'))
        # add custom time ranges
        regridder = xe.Regridder(ds_var, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)
        #
        # with ProgressBar():
        var = regridder(ds_var.tas.resample(time = 'AS-JUN').mean('time')).load()  # var data
        # var = xr.concat([ds_var_hist, ds_var_ssp], dim='time').uas.resample(time = 'AS-JUN').mean('time').load()  # var data
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        # var_anom = funcs.calc_anom_annual(var, var.sel(time = slice('1960', '1990')))
        # var_trend = funcs.calc_trend3d(var_anom.sel(time = slice('1980', '2014')), 'time')
        # var_trend_pval = funcs.calc_trend_pval3d(var_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weightas = np.cos(np.deg2rad(var.lat))

        # print(f'Completed: {model_identifie}')
        # return model_identifier, var_trend, var_trend_pval, gmst_anom, nino34_index, wp_var, ct_var, so_var
        return model_identifier, var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")



In [198]:
models_to_process = [(model) for model in tas_files_cdr if model in tas_files_cdr]
models_to_process

['UKESM1-0-LL_r1i1p1f']

In [199]:
# Run multiprocessing and gather results
res_arr = []
# with mp.Pool(processes=mp.cpu_count()) as pool:
with mp.Pool(processes=4) as pool:
    i = 0
    for res in pool.imap(process_zecmip, models_to_process):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(models_to_process)}', end='\r')
        i += 1



In [200]:
model_list = np.array(res_arr)[:, 0]
model_list

array(['UKESM1-0-LL_r1i1p1f'], dtype=object)

In [201]:
model_var = xr.concat(np.array(res_arr)[:, 1], dim=model_list, coords='minimal', compat='override').rename(dict(concat_dim = 'model')).to_dataset(name = 'tas')

In [202]:
out = xr.merge([model_var])
out

<xarray.Dataset>
Dimensions:             (time: 651, lon: 240, lat: 120, model: 1)
Coordinates:
    height              float64 1.5
  * time                (time) object 1989-06-01 00:00:00 ... 2639-06-01 00:0...
    latitude_longitude  float64 nan
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
  * model               (model) object 'UKESM1-0-LL_r1i1p1f'
Data variables:
    tas                 (model, time, lat, lon) float32 234.7 234.7 ... 260.3

In [203]:
# out.mean('plev').to_netcdf(f'/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/zecmip/{model}_brch1000_co2.nc')
out.to_netcdf(f'/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/cdrmip/{model}_cdr_tas.nc')